<a href="https://colab.research.google.com/github/Shayenvi15/LLM-AI-Projects/blob/main/student_profile_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
!pip install -qU langchain langchain-community langchain-huggingface \
langchain-groq sentence-transformers faiss-cpu gradio PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.2/470.2 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.5/324.5 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/

In [ ]:
# -*- coding: utf-8 -*-
# Core Python modules
import os  # For operating system interactions (file paths, environment variables)
import uuid  # For generating unique identifiers (not currently used)
import tempfile  # For creating temporary files (not currently used)
import fitz  # PyMuPDF library for PDF text extraction and manipulation
import gradio as gr  # For building the web interface
from langchain_groq import ChatGroq  # Integration with Groq's API for LLM access
from langchain.prompts import PromptTemplate  # For creating and managing LLM prompts
from langchain.schema.runnable import RunnableMap  # For creating executable LLM pipelines
from langchain.schema import Document  # Data structure for document handling
from langchain_community.vectorstores import FAISS  # Vector database for similarity search
from langchain_community.embeddings import HuggingFaceEmbeddings  # Text embedding models
from langchain_text_splitters import RecursiveCharacterTextSplitter  # For chunking text documents
from google.colab import userdata  # For securely accessing secrets in Colab environment

# Using Instructor-XL model which understands instructions in embeddings
embedding_model = HuggingFaceEmbeddings(model_name="hkunlp/instructor-xl")

# Large Language Model setup:
# - Uses Groq's ultra-fast Llama3-8b model
# - Temperature 0.3 for balanced creativity/consistency
# - API key securely fetched from Colab's user data secrets
llm = ChatGroq(
    temperature=0.3,
    model_name="llama3-8b-8192",
    api_key=userdata.get("GROQ_API_KEY")  # Securely retrieve API key
)

# Text splitter configuration:
# - Chunk size 1000 characters: Optimal for embedding models
# - Overlap 100 characters: Preserves context between chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

# Stores the raw text of the currently processed resume
current_resume_text = ""

# FAISS vector store instance for similarity search
# Will hold the embedded resume content after processing
vector_store = None

/tmp/ipython-input-2-4851141.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="hkunlp/instructor-xl")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  w

modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.15M [00:00<?, ?B/s]

In [ ]:
def validate_job_title(title):
    """Check if job title is professional and safe"""
    blacklist = {"vigilante", "money laundering", "blackmailer", "terrorist", "poacher"}
    if title.strip().lower() in blacklist:
        return False

    prompt = PromptTemplate.from_template("""
    Strictly Check is '{job_title}' a legitimate professional job title?
    Only accept roles in corporate, technical, academic, or public service domains.
    Answer ONLY 'valid' or 'invalid':
    """)
    chain = prompt | llm
    response = chain.invoke({"job_title": title}).content.strip().lower()
    return "valid" in response

def extract_text_from_pdf(file_path):
    """Improved text extraction with formatting preservation"""
    try:
        doc = fitz.open(file_path)
        text = ""
        for page in doc:
            # Get text with basic formatting
            text += page.get_text("text", sort=True)
        doc.close()
        return text
    except Exception as e:
        raise gr.Error(f"📄 PDF Error: {str(e)}")

def validate_resume(text):
    """More flexible resume validation"""
    section_headers = [
        "education", "skills", "projects", "experience",
        "work history", "achievements", "certifications",
        "qualifications", "internship", "employment"
    ]

    text_lower = text.lower()
    section_count = sum(1 for header in section_headers if header in text_lower)

    if section_count >= 2:
        return True, "Valid resume sections detected"
    return False, f"Only {section_count} resume sections found (need at least 2)"

In [ ]:
def process_resume(file):
    """Processes the uploaded resume file"""
    global current_resume_text, vector_store

    if file is None:
        return "Please upload a PDF file first."

    try:
        file_path = file.name
        current_resume_text = extract_text_from_pdf(file_path)

        is_valid, validation_message = validate_resume(current_resume_text)
        if not is_valid:
            return f"Resume validation failed: {validation_message}"

        # Create embeddings and vector store
        docs = [Document(page_content=current_resume_text)]
        chunks = splitter.split_documents(docs)
        vector_store = FAISS.from_documents(chunks, embedding_model)

        return "✅ Resume processed and ready!"

    except Exception as e:
        return f"An error occurred: {str(e)}"

def generate_cover_letter(student_name, job_title, company):
    """Generate cover letter with validations"""
    global vector_store, current_resume_text

    # Validate inputs
    if not vector_store:
        raise gr.Error("🛠️ Hold up! I need a resume to work my magic. Upload and process one first.")
    if not all([student_name.strip(), job_title.strip(), company.strip()]):
        raise gr.Error("🤷‍♂️ I can't guess your name or job title—fill in all the fields and let's get writing!")
    if not validate_job_title(job_title):
        raise gr.Error(f"🎭 '{job_title}' sounds cool, but HR might prefer 'Software Engineer'. Try again?")

    # Configure retriever
    retriever = vector_store.as_retriever(search_kwargs={"k": 5})

    # Strict RAG prompt
    prompt_template = PromptTemplate.from_template("""
    Create a professional cover letter for {student_name} applying to {company}
    for the position of {job_title}. Use ONLY information from their resume below.

    === RESUME CONTENT ===
    {context}

    STRICT RULES:
    1. Address to Hiring Manager
    2. Highlight ONLY skills/experiences from resume
    3. Use 3-4 paragraphs MAX
    4. Professional tone only
    5. NEVER invent details not in resume
    6. Sign with "{student_name}"
    7. If resume lacks required skills, state: "My background aligns with..."
    8. Do NOT include any type of placeholders such as [Date], [Name], [Address], etc in the cover letter

    Cover Letter:
    """)

    # Construct RAG pipeline
    rag_chain = (
        RunnableMap({
            "context": lambda _: "\n\n".join([
                doc.page_content for doc in retriever.invoke(
                    f"Relevant experience for {job_title} at {company}"
                )
            ]),
            "student_name": lambda _: student_name,
            "job_title": lambda _: job_title,
            "company": lambda _: company
        })
        | prompt_template
        | llm
    )

    # Execute pipeline
    response = rag_chain.invoke({})
    return response.content

def generate_summary():
    """Generates a professional summary from the processed resume"""
    global vector_store, current_resume_text

    if not vector_store:
        raise gr.Error("📄 Please process a resume first!")

    try:
        prompt = PromptTemplate.from_template("""
        Create a professional 3-4 sentence summary of this candidate's qualifications:

        {resume_text}

        Focus on:
        - Key skills and expertise
        - Education background
        - Notable achievements
        - Career objectives
        - Exclude any type of placeholders such as [Date], [Name], [Address], etc

        Summary:
        """)
        chain = prompt | llm
        return chain.invoke({"resume_text": current_resume_text[:3000]}).content
    except Exception as e:
        return f"❌ Error generating summary: {str(e)}"

In [ ]:
# Modified Gradio interface with better error handling
with gr.Blocks(title="Resume to Cover Letter") as app:
    gr.Markdown("📝Cover Letter Generator")

    with gr.Row():
      gr.Markdown()
      file_input = gr.File(
        file_count="single",
        file_types=[".pdf"],
        label="Upload student resume (PDF)")
      with gr.Column():
        process_btn = gr.Button("Process Resume")
        process_status = gr.Textbox(label="Processing Status", interactive=False)


    # Cover Letter Generator
    with gr.Tab("Cover Letter"):
        with gr.Row():
            with gr.Column():
                name_input = gr.Textbox(label="Full Name", placeholder="John Smith")
                job_input = gr.Textbox(label="Job Title", placeholder="Software Engineer")
                company_input = gr.Textbox(label="Company", placeholder="Google")
                generate_btn = gr.Button("Generate Cover Letter", variant="primary")
            with gr.Column():
                output = gr.Textbox(label="Cover Letter", lines=18, interactive=False)

    # Professional Summary tab
    with gr.Tab("Professional Summary"):
        with gr.Row():
            with gr.Column():
                gr.Markdown("### Generate a professional summary from your resume")
                summary_btn = gr.Button("Generate Summary", variant="primary")
            with gr.Column():
                summary_output = gr.Textbox(
                    label="Professional Summary",
                    lines=10,
                    interactive=False,
                    placeholder="Your summary will appear here...")

    # Event Handlers
    process_btn.click(
        process_resume,
        inputs=[file_input],
        outputs=[process_status]
    )
    generate_btn.click(
        generate_cover_letter,
        inputs=[name_input, job_input, company_input],
        outputs=[output]
    )
    summary_btn.click(
        generate_summary,
        inputs=[],
        outputs=[summary_output]
    )

app.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://943025d0ae2718650f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
